# 01. LoL 10분 승패예측 — 데이터 품질 점검 및 전처리

최종 보고서 **2장~3장**의 데이터 구조 파악, 결측/중복 확인, IQR 이상치 점검, `WardsPlaced` 품질 검토,
수학적 중복/선형 종속 검증, 최종 21개 피처 선택을 재현합니다.

실행 결과로 다음 파일을 저장합니다.

- `outputs/data/high_diamond_ranked_10min_preprocessed.csv`
- `outputs/data/game_id_lookup.csv`
- `outputs/tables/iqr_all_numeric.csv`
- `outputs/tables/iqr_report_features.csv`
- `outputs/tables/feature_decisions.csv`

> 핵심 원칙: IQR은 **이상치 후보 탐지**에만 사용합니다. IQR을 벗어났다는 이유만으로 행을 자동 삭제하지 않습니다.


In [1]:
# 필요하면 한 번만 실행
# %pip install pandas numpy

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TARGET = "blueWins"

# VSCode/Jupyter에서 CSV를 노트북과 같은 폴더에 두면 자동으로 찾습니다.
RAW_CANDIDATES = [
    Path("high_diamond_ranked_10min(1).csv"),
    Path("high_diamond_ranked_10min.csv"),
    Path("/mnt/data/high_diamond_ranked_10min(1).csv"),
    Path("/mnt/data/high_diamond_ranked_10min.csv"),
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)

if RAW_PATH is None:
    raise FileNotFoundError(
        "원본 CSV를 찾을 수 없습니다. "
        "high_diamond_ranked_10min(1).csv 또는 high_diamond_ranked_10min.csv를 "
        "노트북과 같은 폴더에 두세요."
    )

DATA_DIR = Path("outputs/data")
TABLE_DIR = Path("outputs/tables")
DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)

print("원본 파일:", RAW_PATH.resolve())
print("shape:", df.shape)
display(df.head())


원본 파일: C:\Users\KTK\Desktop\LOL\high_diamond_ranked_10min.csv
shape: (9879, 40)


,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,blueTowersDestroyed,blueTotalGold,blueAvgLevel,blueTotalExperience,blueTotalMinionsKilled,blueTotalJungleMinionsKilled,blueGoldDiff,blueExperienceDiff,blueCSPerMin,blueGoldPerMin,redWardsPlaced,redWardsDestroyed,redFirstBlood,redKills,redDeaths,redAssists,redEliteMonsters,redDragons,redHeralds,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,0,0,17210,6.6,17039,195,36,643,-8,19.5,1721.0,15,6,0,6,9,8,0,0,0,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,0,0,14712,6.6,16265,174,43,-2908,-1173,17.4,1471.2,12,1,1,5,5,2,2,1,1,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,0,0,16113,6.4,16221,186,46,-1172,-1033,18.6,1611.3,15,3,1,11,7,14,0,0,0,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,1,0,15157,7.0,17954,201,55,-1321,-7,20.1,1515.7,15,2,1,5,4,10,0,0,0,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,0,0,16400,7.0,18543,210,57,-1004,230,21.0,1640.0,17,2,1,6,6,7,1,1,0,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4


## 1. 데이터 구조 및 기본 품질 확인

보고서의 원본 데이터는 **9,879행 × 40열**입니다.
결측치, 완전 중복, `gameId`를 제외한 중복을 각각 확인합니다.


In [2]:
print("행 수:", len(df))
print("열 수:", df.shape[1])
print("\n데이터 타입:")
display(df.dtypes.to_frame("dtype"))

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_rate_pct": df.isna().mean() * 100
})
print("\n결측치 총합:", int(missing["missing_count"].sum()))
display(missing[missing["missing_count"] > 0])

full_duplicates = int(df.duplicated().sum())
duplicates_without_gameid = int(df.drop(columns=["gameId"]).duplicated().sum())

print("완전 중복 행:", full_duplicates)
print("gameId 제외 중복 행:", duplicates_without_gameid)

assert df.shape == (9879, 40)
assert missing["missing_count"].sum() == 0
assert full_duplicates == 0
assert duplicates_without_gameid == 0


행 수: 9879
열 수: 40

데이터 타입:


,dtype
gameId,int64
blueWins,int64
blueWardsPlaced,int64
blueWardsDestroyed,int64
blueFirstBlood,int64
blueKills,int64
blueDeaths,int64
blueAssists,int64
blueEliteMonsters,int64
blueDragons,int64



결측치 총합: 0


,missing_count,missing_rate_pct


완전 중복 행: 0
gameId 제외 중복 행: 0


## 2. 타깃 분포 확인

`blueWins=0`은 Red 승리, `blueWins=1`은 Blue 승리입니다.
두 클래스가 거의 50:50인지 확인합니다.


In [3]:
target_summary = pd.DataFrame({
    "count": df[TARGET].value_counts().sort_index(),
    "rate_pct": (df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)
})
display(target_summary)

assert set(df[TARGET].unique()) == {0, 1}


,count,rate_pct
blueWins,,
0,4949,50.1
1,4930,49.9


## 3. IQR 이상치 후보 탐지

1.5×IQR 기준은 **통계적으로 검토가 필요한 값**을 찾기 위한 도구입니다.
실제 LoL의 스노우볼 상황처럼 정상적인 극단값이 존재할 수 있으므로 자동 삭제 기준으로 사용하지 않습니다.


In [4]:
def iqr_summary(data, columns):
    rows = []
    for col in columns:
        s = data[col].dropna()
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (s < lower) | (s > upper)

        rows.append({
            "feature": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(mask.sum()),
            "outlier_rate_pct": mask.mean() * 100,
            "min": s.min(),
            "max": s.max(),
        })
    return pd.DataFrame(rows)

numeric_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in ["gameId", TARGET]
]
iqr_all = iqr_summary(df, numeric_cols)
iqr_all.to_csv(TABLE_DIR / "iqr_all_numeric.csv", index=False, encoding="utf-8-sig")

report_iqr_features = [
    "blueWardsPlaced", "redWardsPlaced",
    "blueTowersDestroyed", "redTowersDestroyed",
    "blueAssists", "redAssists",
    "blueTotalJungleMinionsKilled", "redTotalJungleMinionsKilled",
    "blueWardsDestroyed", "redWardsDestroyed",
    "blueGoldDiff", "blueExperienceDiff",
    "blueKills", "redKills",
    "blueTotalMinionsKilled", "redTotalMinionsKilled",
]
iqr_report = iqr_summary(df, report_iqr_features)
iqr_report.to_csv(TABLE_DIR / "iqr_report_features.csv", index=False, encoding="utf-8-sig")

display(iqr_report.round(3))


,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_rate_pct,min,max
0,blueWardsPlaced,14.0,20.0,6.0,5.00,29.00,1627,16.469,5,250
1,redWardsPlaced,14.0,20.0,6.0,5.00,29.00,1667,16.874,6,276
2,blueTowersDestroyed,0.0,0.0,0.0,0.00,0.00,464,4.697,0,4
3,redTowersDestroyed,0.0,0.0,0.0,0.00,0.00,396,4.009,0,2
4,blueAssists,4.0,9.0,5.0,-3.50,16.50,209,2.116,0,29
5,redAssists,4.0,9.0,5.0,-3.50,16.50,205,2.075,0,28
6,blueTotalJungleMinionsKilled,44.0,56.0,12.0,26.00,74.00,164,1.660,0,92
7,redTotalJungleMinionsKilled,44.0,57.0,13.0,24.50,76.50,131,1.326,4,92
8,blueWardsDestroyed,1.0,4.0,3.0,-3.50,8.50,136,1.377,0,27
9,redWardsDestroyed,1.0,4.0,3.0,-3.50,8.50,136,1.377,0,24


## 4. `WardsPlaced` 집중 점검

보고서에서는 `WardsPlaced`를 **"게임 시스템상 절대 불가능한 값"**이라고 단정하지 않습니다.

판단 근거는 다음과 같습니다.

- Blue/Red 중앙값은 모두 16인데 최대값은 250/276으로 매우 긴 장꼬리
- IQR 상한 29 초과가 약 16~17%로 단발성 몇 건이 아님
- 100개 이상이 Blue/Red 각각 109건
- 극단값 행의 골드·킬·CS 등 다른 핵심 지표는 정상 범위
- 정확한 수집 패치, 원천 timeline, 집계 로직이 없어 객관적인 clipping 기준을 만들기 어려움
- IQR 초과 행을 삭제하면 너무 많은 경기가 손실됨

따라서 행은 보존하고 `blueWardsPlaced`, `redWardsPlaced` 두 컬럼만 모델 입력에서 제외합니다.


In [5]:
wards = df[["blueWardsPlaced", "redWardsPlaced"]]

display(
    wards.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).T
)

for col in ["blueWardsPlaced", "redWardsPlaced"]:
    s = df[col]
    q1, q3 = s.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)

    print(f"\n[{col}]")
    print("median:", s.median())
    print("IQR upper:", upper)
    print("IQR upper 초과:", int((s > upper).sum()), f"({(s > upper).mean()*100:.2f}%)")
    print(">= 50:", int((s >= 50).sum()))
    print(">= 100:", int((s >= 100).sum()))
    print(">= 150:", int((s >= 150).sum()))
    print("max:", s.max())

# 극단값 행 내부 정합성 예시
extreme_blue = df.loc[df["blueWardsPlaced"].idxmax(), [
    "gameId", "blueWardsPlaced", "blueTotalGold",
    "blueKills", "blueTotalMinionsKilled"
]]
extreme_red = df.loc[df["redWardsPlaced"].idxmax(), [
    "gameId", "redWardsPlaced", "redTotalGold",
    "redKills", "redTotalMinionsKilled"
]]

print("\nBlue WardsPlaced 최대 행:")
display(extreme_blue.to_frame("value"))

print("Red WardsPlaced 최대 행:")
display(extreme_red.to_frame("value"))


,count,mean,std,min,50%,75%,90%,95%,99%,max
blueWardsPlaced,9879.0,22.288288,18.019177,5.0,16.0,20.0,40.0,53.0,105.0,250.0
redWardsPlaced,9879.0,22.367952,18.457427,6.0,16.0,20.0,40.0,53.0,104.0,276.0



[blueWardsPlaced]
median: 16.0
IQR upper: 29.0
IQR upper 초과: 1627 (16.47%)
>= 50: 579
>= 100: 109
>= 150: 19
max: 250

[redWardsPlaced]
median: 16.0
IQR upper: 29.0
IQR upper 초과: 1667 (16.87%)
>= 50: 588
>= 100: 109
>= 150: 27
max: 276

Blue WardsPlaced 최대 행:


,value
gameId,4504166210
blueWardsPlaced,250
blueTotalGold,19704
blueKills,12
blueTotalMinionsKilled,209


Red WardsPlaced 최대 행:


,value
gameId,4495729689
redWardsPlaced,276
redTotalGold,17545
redKills,8
redTotalMinionsKilled,206


## 5. 수학적 중복 및 정확한 선형 종속 검증

보고서에서 제거한 피처는 단순히 상관계수가 높아서 제거한 것이 아니라,
**모든 행에서 정확히 동일하거나 정확한 산술식으로 복원되는 관계**를 직접 검증합니다.


In [6]:
checks = {
    "blueDeaths == redKills":
        (df["blueDeaths"] == df["redKills"]).all(),

    "redDeaths == blueKills":
        (df["redDeaths"] == df["blueKills"]).all(),

    "redFirstBlood == 1 - blueFirstBlood":
        (df["redFirstBlood"] == 1 - df["blueFirstBlood"]).all(),

    "redGoldDiff == -blueGoldDiff":
        (df["redGoldDiff"] == -df["blueGoldDiff"]).all(),

    "redExperienceDiff == -blueExperienceDiff":
        (df["redExperienceDiff"] == -df["blueExperienceDiff"]).all(),

    "blueCSPerMin == blueTotalMinionsKilled / 10":
        np.allclose(df["blueCSPerMin"], df["blueTotalMinionsKilled"] / 10),

    "redCSPerMin == redTotalMinionsKilled / 10":
        np.allclose(df["redCSPerMin"], df["redTotalMinionsKilled"] / 10),

    "blueGoldPerMin == blueTotalGold / 10":
        np.allclose(df["blueGoldPerMin"], df["blueTotalGold"] / 10),

    "redGoldPerMin == redTotalGold / 10":
        np.allclose(df["redGoldPerMin"], df["redTotalGold"] / 10),

    "blueEliteMonsters == blueDragons + blueHeralds":
        (df["blueEliteMonsters"] == df["blueDragons"] + df["blueHeralds"]).all(),

    "redEliteMonsters == redDragons + redHeralds":
        (df["redEliteMonsters"] == df["redDragons"] + df["redHeralds"]).all(),

    "blueGoldDiff == blueTotalGold - redTotalGold":
        (df["blueGoldDiff"] == df["blueTotalGold"] - df["redTotalGold"]).all(),

    "blueExperienceDiff == blueTotalExperience - redTotalExperience":
        (df["blueExperienceDiff"] ==
         df["blueTotalExperience"] - df["redTotalExperience"]).all(),
}

check_df = pd.DataFrame(
    [{"relationship": k, "holds_for_all_rows": bool(v)} for k, v in checks.items()]
)
display(check_df)

assert check_df["holds_for_all_rows"].all()


,relationship,holds_for_all_rows
0,blueDeaths == redKills,True
1,redDeaths == blueKills,True
2,redFirstBlood == 1 - blueFirstBlood,True
3,redGoldDiff == -blueGoldDiff,True
4,redExperienceDiff == -blueExperienceDiff,True
5,blueCSPerMin == blueTotalMinionsKilled / 10,True
6,redCSPerMin == redTotalMinionsKilled / 10,True
7,blueGoldPerMin == blueTotalGold / 10,True
8,redGoldPerMin == redTotalGold / 10,True
9,blueEliteMonsters == blueDragons + blueHeralds,True


## 6. 최종 피처 선택

최종 모델에는 21개 피처를 사용합니다.

`TotalGold/TotalExperience` 대신 `GoldDiff/ExperienceDiff`를 선택하는 것은
정보 손실이 전혀 없는 제거가 아닙니다. 절대 수준 정보는 일부 축약되지만,
프로젝트의 질문인 **"10분 시점에서 상대보다 얼마나 앞서 있는가"**를 직접 표현하고
정확한 선형 종속을 피하기 위한 의도적 선택입니다.


In [7]:
FINAL_FEATURES = [
    "blueWardsDestroyed",
    "blueKills",
    "blueAssists",
    "blueFirstBlood",
    "blueDragons",
    "blueHeralds",
    "blueTowersDestroyed",
    "blueAvgLevel",
    "blueTotalMinionsKilled",
    "blueTotalJungleMinionsKilled",
    "blueGoldDiff",
    "blueExperienceDiff",
    "redWardsDestroyed",
    "redKills",
    "redAssists",
    "redDragons",
    "redHeralds",
    "redTowersDestroyed",
    "redAvgLevel",
    "redTotalMinionsKilled",
    "redTotalJungleMinionsKilled",
]

REMOVE_REASONS = {
    "gameId": "고유 식별자: 일반화 가능한 경기 상태 정보가 아님",
    "blueWardsPlaced": "반복적 극단 장꼬리와 원천 집계 정의 불확실성",
    "redWardsPlaced": "반복적 극단 장꼬리와 원천 집계 정의 불확실성",
    "blueDeaths": "redKills와 정확히 동일",
    "redDeaths": "blueKills와 정확히 동일",
    "redFirstBlood": "1 - blueFirstBlood로 정확히 복원 가능",
    "redGoldDiff": "-blueGoldDiff로 정확히 복원 가능",
    "redExperienceDiff": "-blueExperienceDiff로 정확히 복원 가능",
    "blueCSPerMin": "blueTotalMinionsKilled / 10",
    "redCSPerMin": "redTotalMinionsKilled / 10",
    "blueGoldPerMin": "blueTotalGold / 10",
    "redGoldPerMin": "redTotalGold / 10",
    "blueEliteMonsters": "blueDragons + blueHeralds; 세부 오브젝트 유지",
    "redEliteMonsters": "redDragons + redHeralds; 세부 오브젝트 유지",
    "blueTotalGold": "GoldDiff 중심 표현 선택으로 선형 종속 해소",
    "redTotalGold": "GoldDiff 중심 표현 선택으로 선형 종속 해소",
    "blueTotalExperience": "ExperienceDiff 중심 표현 선택으로 선형 종속 해소",
    "redTotalExperience": "ExperienceDiff 중심 표현 선택으로 선형 종속 해소",
}

decision_rows = []
for col in df.columns:
    if col == TARGET:
        decision, reason = "TARGET", "최종 승패 라벨"
    elif col in FINAL_FEATURES:
        decision, reason = "KEEP", "최종 모델 입력 피처"
    else:
        decision, reason = "REMOVE", REMOVE_REASONS.get(col, "검토 필요")

    decision_rows.append({
        "feature": col,
        "decision": decision,
        "reason": reason
    })

feature_decisions = pd.DataFrame(decision_rows)
display(feature_decisions)

assert len(FINAL_FEATURES) == 21
assert not (feature_decisions["reason"] == "검토 필요").any()

feature_decisions.to_csv(
    TABLE_DIR / "feature_decisions.csv",
    index=False,
    encoding="utf-8-sig"
)


,feature,decision,reason
0,gameId,REMOVE,고유 식별자: 일반화 가능한 경기 상태 정보가 아님
1,blueWins,TARGET,최종 승패 라벨
2,blueWardsPlaced,REMOVE,반복적 극단 장꼬리와 원천 집계 정의 불확실성
3,blueWardsDestroyed,KEEP,최종 모델 입력 피처
4,blueFirstBlood,KEEP,최종 모델 입력 피처
5,blueKills,KEEP,최종 모델 입력 피처
6,blueDeaths,REMOVE,redKills와 정확히 동일
7,blueAssists,KEEP,최종 모델 입력 피처
8,blueEliteMonsters,REMOVE,blueDragons + blueHeralds; 세부 오브젝트 유지
9,blueDragons,KEEP,최종 모델 입력 피처


## 7. 전처리 데이터 저장 및 재검증

행은 삭제하지 않고 최종 21개 피처와 타깃만 저장합니다.
`gameId`는 모델에 넣지 않지만, 오분류 사례를 추적하기 위해 별도 lookup 파일에 보존합니다.


In [8]:
preprocessed = df[FINAL_FEATURES + [TARGET]].copy()

PREPROCESSED_PATH = DATA_DIR / "high_diamond_ranked_10min_preprocessed.csv"
LOOKUP_PATH = DATA_DIR / "game_id_lookup.csv"

preprocessed.to_csv(PREPROCESSED_PATH, index=False, encoding="utf-8-sig")

lookup = pd.DataFrame({
    "row_index": np.arange(len(df)),
    "gameId": df["gameId"].values
})
lookup.to_csv(LOOKUP_PATH, index=False, encoding="utf-8-sig")

# 저장 후 재로드하여 스키마 검증
reloaded = pd.read_csv(PREPROCESSED_PATH)

assert reloaded.shape == (9879, 22)
assert reloaded.columns.tolist() == FINAL_FEATURES + [TARGET]
assert reloaded.isna().sum().sum() == 0
assert reloaded.duplicated().sum() == 0
assert set(reloaded[TARGET].unique()) == {0, 1}

print("전처리 완료:", PREPROCESSED_PATH.resolve())
print("gameId lookup:", LOOKUP_PATH.resolve())
print("최종 shape:", reloaded.shape)


전처리 완료: C:\Users\KTK\Desktop\LOL\outputs\data\high_diamond_ranked_10min_preprocessed.csv
gameId lookup: C:\Users\KTK\Desktop\LOL\outputs\data\game_id_lookup.csv
최종 shape: (9879, 22)
